In [ ]:
import geopandas as gpd
import pandas as pd

# ============================================================
# INPUT
# ============================================================

GPKG = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"

LAYERS = [
    "garissa_with_25d",
    "kigali_with_25d",
    "juba_with_25d",
    "tripoli_with_25d",
    "dakar_with_25d",
    "johannesburg_with_25d",
]

OUTPUT_LAYER = "all_buildings_with_25d"

# Common CRS for all layers
TARGET_CRS = "EPSG:4326"


# ============================================================
# READ 6 LAYERS
# ============================================================

gdfs = []

for layer in LAYERS:

    print(f"\nReading: {layer}")

    gdf = gpd.read_file(
        GPKG,
        layer=layer,
        engine="pyogrio"
    )

    print(f"  Rows: {len(gdf):,}")
    print(f"  Original CRS: {gdf.crs}")

    # --------------------------------------------------------
    # Reproject to common CRS
    # --------------------------------------------------------

    if gdf.crs is None:
        raise ValueError(
            f"Layer '{layer}' does not have a CRS."
        )

    gdf = gdf.to_crs(TARGET_CRS)

    print(f"  Reprojected to: {TARGET_CRS}")

    gdfs.append(gdf)

# ============================================================
# FIND FIELDS PRESENT IN ALL 6 LAYERS
# ============================================================

field_sets = [
    set(gdf.columns)
    for gdf in gdfs
]

common_fields = set.intersection(
    *field_sets
)

# Fields that are not present in all 6
all_fields = set.union(
    *field_sets
)

dropped_fields = sorted(
    all_fields - common_fields
)


# ============================================================
# PRINT DROPPED FIELDS
# ============================================================

if dropped_fields:

    print("\n==========================================")
    print("FIELDS DROPPED")
    print("==========================================")

    for field in dropped_fields:
        print(f"  Dropped: {field}")

else:

    print("\nNo fields were dropped.")


print(
    f"\nFields retained: {len(common_fields)}"
)


# ============================================================
# KEEP ONLY COMMON FIELDS
# ============================================================

# Preserve the original column order from the first layer
common_fields_ordered = [
    col
    for col in gdfs[0].columns
    if col in common_fields
]

gdfs_clean = [
    gdf[common_fields_ordered].copy()
    for gdf in gdfs
]


# ============================================================
# MERGE ALL 6
# ============================================================

merged_gdf = gpd.GeoDataFrame(
    pd.concat(
        gdfs_clean,
        ignore_index=True
    ),
    crs=gdfs[0].crs
)


# ============================================================
# PRINT FINAL ROW COUNT
# ============================================================

print("\n==========================================")
print("MERGE COMPLETE")
print("==========================================")

print(
    f"Final number of rows: {len(merged_gdf):,}"
)

print(
    f"Final number of fields: {len(merged_gdf.columns)}"
)

print(
    f"CRS: {merged_gdf.crs}"
)


# ============================================================
# SAVE
# ============================================================

merged_gdf.to_file(
    GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG",
    engine="pyogrio"
)

print(
    f"\nSaved to layer: {OUTPUT_LAYER}"
)


Outlier analysis dropped 73519 buildings out of 10919172
Google 2.5D analysis dropped an additional 797289 buildings out of 10845653


In [3]:
# Find and drop false positives

gdf = gpd.read_file(r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb", layer= "all_buildings_with_25d")

# Check 1:
# If isolation_forest_outlier == True and geometry_confidence != 'high' and top_class_name != 'built', or  top_probability < .6
orig_len = (len(gdf))

drop_mask = (
    (
        (gdf["isolation_forest_outlier"] == True) &
        (gdf["geometry_confidence"] != "high") &
        (gdf["top_class_name"] != "built")
    )
    |
    (
        (gdf["top_class_name"] == "built") &
        (gdf["top_probability"] < 0.6)
    )
)

gdf = gdf[~drop_mask].copy()

outlier_len = len(gdf)

dif1 = orig_len - outlier_len


print(f"Outlier analysis dropped {dif1} buildings out of {orig_len}")

# Check 2: If google25d_max_confidence < 0.1 and geometry_provider != 'OSM' and year > 2023

drop_mask = (
    (gdf["google25d_max_confidence"] < 0.1) &
    (
        (gdf["geometry_provider"] != "OSM") |
        (gdf["year"] > 2023)
    )
)

gdf = gdf[~drop_mask].copy()

g25_len = len(gdf)

dif2 = outlier_len - g25_len

print(f"Google 2.5D analysis dropped an additional {dif2} buildings out of {outlier_len}")

Outlier analysis dropped 0 buildings out of 11655267
Google 2.5D analysis dropped an additional 977238 buildings out of 11655267


In [10]:
# Seperate GDF based on source
gdf_ov = gdf[gdf['geometry_provider']!= "3D-GloBFP"]
gdf_glo = gdf[gdf['geometry_provider']== "3D-GloBFP"]

gdf_ov.to_file(r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb", layer="all_buildings_25_removed_overture", driver="OpenFileGDB", engine="pyogrio")
gdf_glo.to_file(r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb", layer="all_buildings_25_removed_glo", driver="OpenFileGDB", engine="pyogrio")

In [11]:
# Remove overlaps with 10% IoU

import arcpy
import os

# ============================================================
# SETTINGS
# ============================================================

GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

GLO = os.path.join(
    GDB,
    "all_buildings_25_removed_glo"
)

OV = os.path.join(
    GDB,
    "all_buildings_25_removed_overture"
)

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------

OV_KEEP = os.path.join(
    GDB,
    "all_buildings_25d_ov_keep"
)

GLO_KEEP = os.path.join(
    GDB,
    "all_buildings_25d_glo_keep"
)

# ------------------------------------------------------------
# IoU threshold
# ------------------------------------------------------------

IOU_THRESHOLD = 0.10


# ============================================================
# DELETE EXISTING OUTPUTS
# ============================================================

print("=" * 70)
print("PREPARING OUTPUTS")
print("=" * 70)

for fc in [OV_KEEP, GLO_KEEP]:

    if arcpy.Exists(fc):

        print(f"Deleting existing output: {fc}")

        arcpy.management.Delete(fc)


# ============================================================
# COPY INPUTS
# ============================================================

print()
print("Copying input feature classes...")

arcpy.management.CopyFeatures(
    OV,
    OV_KEEP
)

arcpy.management.CopyFeatures(
    GLO,
    GLO_KEEP
)

print("Overture copied.")
print("Glo copied.")


# ============================================================
# ADD UNIQUE IDS
# ============================================================

def add_unique_id(fc, field_name):

    fields = [
        f.name.lower()
        for f in arcpy.ListFields(fc)
    ]

    if field_name.lower() not in fields:

        arcpy.management.AddField(
            fc,
            field_name,
            "LONG"
        )

    i = 1

    with arcpy.da.UpdateCursor(
        fc,
        [field_name]
    ) as cursor:

        for row in cursor:

            row[0] = i

            cursor.updateRow(row)

            i += 1


print()
print("=" * 70)
print("CREATING UNIQUE IDS")
print("=" * 70)

add_unique_id(
    OV_KEEP,
    "OV_ID"
)

add_unique_id(
    GLO_KEEP,
    "GLO_ID"
)


# ============================================================
# CREATE AREA LOOKUPS
# ============================================================

print()
print("=" * 70)
print("CALCULATING BUILDING AREAS")
print("=" * 70)

ov_area = {}

with arcpy.da.SearchCursor(
    OV_KEEP,
    [
        "OV_ID",
        "SHAPE@AREA"
    ]
) as cursor:

    for ov_id, area in cursor:

        if area is not None and area > 0:

            ov_area[ov_id] = float(area)


glo_area = {}

with arcpy.da.SearchCursor(
    GLO_KEEP,
    [
        "GLO_ID",
        "SHAPE@AREA"
    ]
) as cursor:

    for glo_id, area in cursor:

        if area is not None and area > 0:

            glo_area[glo_id] = float(area)


print(
    f"Overture buildings with valid area: "
    f"{len(ov_area):,}"
)

print(
    f"Glo buildings with valid area: "
    f"{len(glo_area):,}"
)


# ============================================================
# CREATE INTERSECTION
# ============================================================

print()
print("=" * 70)
print("CREATING OVERTURE / GLO INTERSECTIONS")
print("=" * 70)

overlap_table = os.path.join(
    GDB,
    "ov_glo_intersections_method2_tmp"
)

if arcpy.Exists(overlap_table):

    arcpy.management.Delete(
        overlap_table
    )


arcpy.analysis.Intersect(
    [
        OV_KEEP,
        GLO_KEEP
    ],
    overlap_table,
    "ALL",
    output_type="INPUT"
)

print("Intersection complete.")


# ============================================================
# IDENTIFY ID FIELDS
# ============================================================

intersection_fields = [
    f.name
    for f in arcpy.ListFields(
        overlap_table
    )
]

print()
print("Intersection fields:")

for field in intersection_fields:

    print(
        f"  {field}"
    )


field_lookup = {
    f.name.lower(): f.name
    for f in arcpy.ListFields(
        overlap_table
    )
}

ov_id_field = field_lookup.get(
    "ov_id"
)

glo_id_field = field_lookup.get(
    "glo_id"
)


if ov_id_field is None:

    raise RuntimeError(
        "Could not find OV_ID in intersection output."
    )


if glo_id_field is None:

    raise RuntimeError(
        "Could not find GLO_ID in intersection output."
    )


print()
print(
    f"Overture ID field: {ov_id_field}"
)

print(
    f"Glo ID field:      {glo_id_field}"
)


# ============================================================
# EVALUATE IoU AND CONTAINMENT
# ============================================================

print()
print("=" * 70)
print("EVALUATING IoU AND CONTAINMENT")
print("=" * 70)

drop_ov = set()

total_intersections = 0
iou_matches = 0
contained_matches = 0


# ------------------------------------------------------------
# Iterate through all Overture / Glo intersections
# ------------------------------------------------------------

with arcpy.da.SearchCursor(
    overlap_table,
    [
        ov_id_field,
        glo_id_field,
        "SHAPE@AREA"
    ]
) as cursor:

    for row in cursor:

        ov_id = row[0]
        glo_id = row[1]
        intersection_area = row[2]

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        if ov_id is None:
            continue

        if glo_id is None:
            continue

        if intersection_area is None:
            continue

        if intersection_area <= 0:
            continue

        # ----------------------------------------------------
        # Original polygon areas
        # ----------------------------------------------------

        ov_original_area = ov_area.get(
            ov_id
        )

        glo_original_area = glo_area.get(
            glo_id
        )

        if ov_original_area is None:
            continue

        if glo_original_area is None:
            continue

        if ov_original_area <= 0:
            continue

        if glo_original_area <= 0:
            continue

        total_intersections += 1

        # ====================================================
        # CALCULATE IoU
        # ====================================================

        union_area = (
            ov_original_area
            +
            glo_original_area
            -
            intersection_area
        )

        if union_area <= 0:
            continue

        iou = (
            intersection_area
            /
            union_area
        )

        # ====================================================
        # CONDITION 1:
        #
        # IoU > 10%
        # ====================================================

        iou_match = (
            iou > IOU_THRESHOLD
        )

        # ====================================================
        # CONDITION 2:
        #
        # Overture completely within Glo
        # ====================================================

        area_tolerance = 0.000001

        contained = (
            intersection_area
            >=
            ov_original_area * (
                1 - area_tolerance
            )
        )

        # ====================================================
        # DROP OVERTURE IF EITHER CONDITION IS TRUE
        # ====================================================

        if iou_match:

            drop_ov.add(
                ov_id
            )

            iou_matches += 1

        elif contained:

            drop_ov.add(
                ov_id
            )

            contained_matches += 1


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 70)
print("OVERLAP RESULTS")
print("=" * 70)

print(
    f"Overture/Glo intersecting pairs: "
    f"{total_intersections:,}"
)

print(
    f"Overture buildings meeting IoU > "
    f"{IOU_THRESHOLD:.0%}: "
    f"{iou_matches:,}"
)

print(
    f"Overture buildings fully within Glo: "
    f"{contained_matches:,}"
)

print(
    f"Unique Overture buildings to remove: "
    f"{len(drop_ov):,}"
)


# ============================================================
# DELETE OVERTURE BUILDINGS
# ============================================================

print()
print("=" * 70)
print("REMOVING OVERTURE BUILDINGS")
print("=" * 70)

deleted_ov = 0

with arcpy.da.UpdateCursor(
    OV_KEEP,
    ["OV_ID"]
) as cursor:

    for row in cursor:

        ov_id = row[0]

        if ov_id in drop_ov:

            cursor.deleteRow()

            deleted_ov += 1


print(
    f"Deleted Overture buildings: "
    f"{deleted_ov:,}"
)

print(
    "Glo buildings deleted: 0"
)


# ============================================================
# CLEAN UP TEMPORARY ID FIELDS
# ============================================================

print()
print("=" * 70)
print("CLEANING UP")
print("=" * 70)


for fc, field_name in [
    (OV_KEEP, "OV_ID"),
    (GLO_KEEP, "GLO_ID")
]:

    existing_fields = [
        f.name
        for f in arcpy.ListFields(fc)
    ]

    if field_name in existing_fields:

        arcpy.management.DeleteField(
            fc,
            field_name
        )


# ============================================================
# DELETE TEMPORARY INTERSECTION
# ============================================================

if arcpy.Exists(
    overlap_table
):

    arcpy.management.Delete(
        overlap_table
    )


# ============================================================
# FINAL COUNTS
# ============================================================

final_ov_count = int(
    arcpy.management.GetCount(
        OV_KEEP
    )[0]
)

final_glo_count = int(
    arcpy.management.GetCount(
        GLO_KEEP
    )[0]
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Original Overture buildings: "
    f"{len(ov_area):,}"
)

print(
    f"Original Glo buildings:      "
    f"{len(glo_area):,}"
)

print()

print(
    f"Overture removed:             "
    f"{deleted_ov:,}"
)

print(
    f"Glo removed:                  "
    f"0"
)

print()

print(
    f"FINAL Overture buildings:     "
    f"{final_ov_count:,}"
)

print(
    f"FINAL Glo buildings:          "
    f"{final_glo_count:,}"
)

print()
print("OUTPUTS")
print("-------")

print(
    f"Overture: {OV_KEEP}"
)

print(
    f"Glo:      {GLO_KEEP}"
)

print()
print("=" * 70)
print("DONE")
print("=" * 70)

PREPARING OUTPUTS

Copying input feature classes...
Overture copied.
Glo copied.

CREATING UNIQUE IDS

CALCULATING BUILDING AREAS
Overture buildings with valid area: 8,807,557
Glo buildings with valid area: 1,870,472

CREATING OVERTURE / GLO INTERSECTIONS
Intersection complete.

Intersection fields:
  OBJECTID
  SHAPE
  FID_all_buildings_25d_ov_keep
  OBJECTID_1
  geometry_source
  geometry_source_id
  geometry_provider
  geometry_dataset
  geometry_license
  source_record_id
  source_update_time
  source_version
  native_height_m
  native_floors
  area_m2
  integrated_id
  globfp_support
  osm_version
  osm_multiple_versions
  height_globfp_vector_m
  globfp_height_id
  height_floors_estimate_m
  height_best_m
  height_source
  height_confidence
  geometry_confidence
  selection_reason
  review_required
  ANALYSIS_ID
  SEGMENT_UID
  height_google_30m_m
  height_globfp_grid_30m_m
  height_tempo_100m_m
  height_wsf3d_100m_m
  wsf2019_settlement_fraction
  height_source_count
  height_ra

In [ ]:
# Merge back

import arcpy
import os

# ============================================================
# INPUT GEODATABASE
# ============================================================

GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

OV = os.path.join(
    GDB,
    "all_buildings_25d_ov_keep"
)

GLO = os.path.join(
    GDB,
    "all_buildings_25d_glo_keep"
)

# ------------------------------------------------------------
# OUTPUT GEOPACKAGE
# ------------------------------------------------------------

OUT_GPKG = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"

OUT_LAYER = "all_buildings_25d_cleaned_integrated"

OUT_FC = os.path.join(
    OUT_GPKG,
    OUT_LAYER
)

# ============================================================
# CHECK INPUTS
# ============================================================

print("=" * 70)
print("MERGING CLEANED BUILDING LAYERS")
print("=" * 70)

if not arcpy.Exists(GLO):
    raise FileNotFoundError(
        f"GloBFP input not found:\n{GLO}"
    )

if not arcpy.Exists(OV):
    raise FileNotFoundError(
        f"Overture input not found:\n{OV}"
    )

# ============================================================
# CHECK / CREATE OUTPUT DIRECTORY
# ============================================================

out_folder = os.path.dirname(OUT_GPKG)

if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# ============================================================
# CREATE GEOPACKAGE
# ============================================================

if not arcpy.Exists(OUT_GPKG):
    print("\nCreating GeoPackage...")
    
    arcpy.management.CreateSQLiteDatabase(
        OUT_GPKG,
        "GEOPACKAGE"
    )

else:
    print("\nGeoPackage already exists.")

# ============================================================
# DELETE EXISTING OUTPUT
# ============================================================

if arcpy.Exists(OUT_FC):
    print("\nDeleting existing output layer...")
    arcpy.management.Delete(OUT_FC)

# ============================================================
# REPORT INPUT COUNTS
# ============================================================

glo_count = int(arcpy.management.GetCount(GLO)[0])
ov_count = int(arcpy.management.GetCount(OV)[0])

print("\nInput feature counts:")
print(f"  GloBFP:    {glo_count:,}")
print(f"  Overture:  {ov_count:,}")
print(f"  Total:     {glo_count + ov_count:,}")

# ============================================================
# MERGE
# ============================================================

print("\nMerging layers...")

arcpy.management.Merge(
    [GLO, OV],
    OUT_FC
)

# ============================================================
# VERIFY OUTPUT
# ============================================================

out_count = int(arcpy.management.GetCount(OUT_FC)[0])

print("\n" + "=" * 70)
print("MERGE COMPLETE")
print("=" * 70)

print(f"GloBFP features:       {glo_count:,}")
print(f"Overture features:     {ov_count:,}")
print(f"Expected total:        {glo_count + ov_count:,}")
print(f"Output features:       {out_count:,}")

print("\nOutput:")
print(OUT_FC)

if out_count == glo_count + ov_count:
    print("\nFeature count check: PASSED")
else:
    print("\nWARNING: Output count differs from expected total.")

print("\nDone.")

MERGING CLEANED BUILDING LAYERS

GeoPackage already exists.

Input feature counts:
  GloBFP:    1,870,472
  Overture:  8,478,317
  Total:     10,348,789

Merging layers...

MERGE COMPLETE
GloBFP features:       1,870,472
Overture features:     8,478,317
Expected total:        10,348,789
Output features:       10,348,789

Output:
C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\all_buildings_25d_cleaned_integrated

Feature count check: PASSED

Done.


In [1]:
import geopandas as gpd

# ============================================================
# INPUTS
# ============================================================

BUILDINGS_GPKG = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
BUILDINGS_LAYER = "all_buildings_25d_cleaned_integrated"

POLYGONS_GPKG = r"E:\buildings\cleaning\GHS_UCDB_REGION_SUB_SAHARAN_AFRICA_R2024A_V1_2\GHS_UCDB_REGION_SUB_SAHARAN_AFRICA_R2024A.gpkg"
POLYGONS_LAYER = "GHSL_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A"

OUTPUT_GPKG = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
OUTPUT_LAYER = "all_buildings_25d_cleaned_integrated_small_cleaned"


# ============================================================
# READ DATA
# ============================================================

buildings = gpd.read_file(
    BUILDINGS_GPKG,
    layer=BUILDINGS_LAYER
)

polygons = gpd.read_file(
    POLYGONS_GPKG,
    layer=POLYGONS_LAYER
)


# ============================================================
# MAKE SURE CRS MATCHES
# ============================================================

if buildings.crs != polygons.crs:
    polygons = polygons.to_crs(buildings.crs)


# ============================================================
# FIND BUILDINGS THAT OVERLAP POLYGONS
# ============================================================

# Spatial join: buildings that intersect at least one polygon
overlapping = gpd.sjoin(
    buildings[["geometry"]],
    polygons[["geometry"]],
    how="left",
    predicate="intersects"
)

# Building indices that overlap at least one polygon
overlapping_ids = overlapping.loc[
    overlapping["index_right"].notna()
].index.unique()


# ============================================================
# IDENTIFY BUILDINGS TO REMOVE
# ============================================================

# Building overlaps a polygon
overlapping_polygon = buildings.index.isin(overlapping_ids)

# Building is smaller than 16 m²
small_building = buildings["area_m2"] < 16

# REMOVE ONLY WHEN BOTH CONDITIONS ARE TRUE
remove = overlapping_polygon & small_building


# ============================================================
# CREATE CLEAN DATASET
# ============================================================

buildings_cleaned = buildings.loc[~remove].copy()


# ============================================================
# OUTPUT
# ============================================================

buildings_cleaned.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="OpenFileGDB"
)


# ============================================================
# SUMMARY
# ============================================================

print(f"Original buildings: {len(buildings):,}")
print(f"Small buildings overlapping polygons: {remove.sum():,}")
print(f"Remaining buildings: {len(buildings_cleaned):,}")
print(f"Output: {OUTPUT_GPKG}")

Original buildings: 10,348,789
Small buildings overlapping polygons: 983,410
Remaining buildings: 9,365,379
Output: C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb


In [2]:
# Remove nesting buildings

import geopandas as gpd

# ============================================================
# INPUTS
# ============================================================

INPUT_GPKG = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
INPUT_LAYER = "all_buildings_25d_cleaned_integrated_small_cleaned"

OUTPUT_GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
OUTPUT_LAYER = "all_buildings_25d_cleaned_integrated_small_cleaned_unested"


# ============================================================
# READ BUILDINGS
# ============================================================

buildings = gpd.read_file(
    INPUT_GPKG,
    layer=INPUT_LAYER
)

print(f"Original buildings: {len(buildings):,}")


# ============================================================
# FIND BUILDINGS COMPLETELY CONTAINED BY ANOTHER BUILDING
# ============================================================

# Find polygons that are completely within another polygon
contained = gpd.sjoin(
    buildings[["geometry"]],
    buildings[["geometry"]],
    how="inner",
    predicate="within",
    lsuffix="contained",
    rsuffix="containing"
)

# Remove self-matches
contained = contained[
    contained.index != contained["index_containing"]
]


# ============================================================
# IDENTIFY LARGER CONTAINING BUILDINGS TO REMOVE
# ============================================================

remove_ids = set()

for contained_id, row in contained.iterrows():

    containing_id = row["index_containing"]

    contained_area = buildings.loc[
        contained_id, "geometry"
    ].area

    containing_area = buildings.loc[
        containing_id, "geometry"
    ].area

    # Remove the larger polygon
    if containing_area > contained_area:
        remove_ids.add(containing_id)


# ============================================================
# REMOVE LARGER CONTAINING BUILDINGS
# ============================================================

buildings_cleaned = buildings.loc[
    ~buildings.index.isin(remove_ids)
].copy()


# ============================================================
# EXPORT TO FILE GEODATABASE
# ============================================================

buildings_cleaned.to_file(
    OUTPUT_GDB,
    layer=OUTPUT_LAYER,
    driver="OpenFileGDB"
)


# ============================================================
# SUMMARY
# ============================================================

print(f"Buildings removed: {len(remove_ids):,}")
print(f"Buildings remaining: {len(buildings_cleaned):,}")
print(f"Output: {OUTPUT_GDB}")
print(f"Layer: {OUTPUT_LAYER}")

Original buildings: 9,365,379
Buildings removed: 378
Buildings remaining: 9,365,001
Output: C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Layer: all_buildings_25d_cleaned_integrated_small_cleaned_unested


In [3]:
# Generate neighbor stats

import geopandas as gpd
import numpy as np
from shapely.strtree import STRtree

# ============================================================
# INPUT / OUTPUT
# ============================================================

INPUT_GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
INPUT_LAYER = "all_buildings_25d_cleaned_integrated_small_cleaned_unested"

OUTPUT_GDB = r"C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb"
OUTPUT_LAYER = "all_buildings_25d_cleaned_integrated_small_cleaned_unested"


# ============================================================
# PARAMETERS
# ============================================================

DISTANCE_THRESHOLD = 50.0  # meters


# ============================================================
# READ BUILDINGS
# ============================================================

gdf = gpd.read_file(
    INPUT_GDB,
    layer=INPUT_LAYER
)

print(f"Buildings: {len(gdf):,}")


# ============================================================
# PROJECT TO A CRS IN METERS
# ============================================================

# If already projected, keep the CRS.
# Otherwise automatically use an appropriate local UTM CRS.
if gdf.crs.is_geographic:
    metric_crs = gdf.estimate_utm_crs()
    gdf = gdf.to_crs(metric_crs)

print(f"Distance CRS: {gdf.crs}")


# ============================================================
# CALCULATE GEOMETRIC AREA
# ============================================================

# Use existing area_m2 if available.
# Otherwise calculate area from the projected geometry.

if "area_m2" not in gdf.columns:
    gdf["area_m2"] = gdf.geometry.area


# ============================================================
# SPATIAL INDEX
# ============================================================

geometries = gdf.geometry.values

tree = STRtree(geometries)


# ============================================================
# CALCULATE NEIGHBOR STATISTICS
# ============================================================

nn_size_mean = []
nn_size_min = []
nn_size_max = []

nn_dist_mean = []
nn_dist_min = []
nn_dist_max = []

nn_count = []


for i, geom in enumerate(geometries):

    # --------------------------------------------------------
    # Find candidate buildings whose bounding boxes are
    # within 50 m of this building
    # --------------------------------------------------------

    search_geom = geom.buffer(DISTANCE_THRESHOLD)

    candidate_indices = tree.query(
        search_geom,
        predicate="intersects"
    )

    neighbor_sizes = []
    neighbor_distances = []

    for j in candidate_indices:

        # Don't compare building with itself
        if j == i:
            continue

        neighbor_geom = geometries[j]

        # ----------------------------------------------------
        # Actual edge-to-edge distance
        # ----------------------------------------------------

        distance = geom.distance(neighbor_geom)

        # Keep only buildings within 50 m
        if distance <= DISTANCE_THRESHOLD:

            neighbor_sizes.append(
                gdf.iloc[j]["area_m2"]
            )

            neighbor_distances.append(
                distance
            )

    # --------------------------------------------------------
    # Neighbor size statistics
    # --------------------------------------------------------

    if neighbor_sizes:

        nn_size_mean.append(
            np.mean(neighbor_sizes)
        )

        nn_size_min.append(
            np.min(neighbor_sizes)
        )

        nn_size_max.append(
            np.max(neighbor_sizes)
        )

        # ----------------------------------------------------
        # Edge distance statistics
        # ----------------------------------------------------

        nn_dist_mean.append(
            np.mean(neighbor_distances)
        )

        nn_dist_min.append(
            np.min(neighbor_distances)
        )

        nn_dist_max.append(
            np.max(neighbor_distances)
        )

        nn_count.append(
            len(neighbor_sizes)
        )

    else:

        # No neighbors within 50 m
        nn_size_mean.append(np.nan)
        nn_size_min.append(np.nan)
        nn_size_max.append(np.nan)

        nn_dist_mean.append(np.nan)
        nn_dist_min.append(np.nan)
        nn_dist_max.append(np.nan)

        nn_count.append(0)


# ============================================================
# ADD VARIABLES
# ============================================================

gdf["nn_size_mean"] = nn_size_mean
gdf["nn_size_min"] = nn_size_min
gdf["nn_size_max"] = nn_size_max

gdf["nn_edge_dist_mean_m"] = nn_dist_mean
gdf["nn_edge_dist_min_m"] = nn_dist_min
gdf["nn_edge_dist_max_m"] = nn_dist_max

gdf["nn_count_50m"] = nn_count


# ============================================================
# EXPORT
# ============================================================

gdf.to_file(
    OUTPUT_GDB,
    layer=OUTPUT_LAYER,
    driver="OpenFileGDB"
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("Completed.")
print(f"Output: {OUTPUT_GDB}")
print(f"Layer: {OUTPUT_LAYER}")
print()
print(f"Buildings with ≥1 neighbor within 50 m: {(gdf['nn_count_50m'] > 0).sum():,}")
print(f"Buildings with no neighbor within 50 m:  {(gdf['nn_count_50m'] == 0).sum():,}")

Buildings: 9,365,001
Distance CRS: EPSG:32632


C:\Users\msw2198\AppData\Roaming\Python\Python313\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field nn_count_50m of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(



Completed.
Output: C:\Users\msw2198\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Layer: all_buildings_25d_cleaned_integrated_small_cleaned_unested

Buildings with ≥1 neighbor within 50 m: 9,315,499
Buildings with no neighbor within 50 m:  49,502
